# Clasificación de la expresión a partir del clip

Tercera etapa. Toma los clips de 16 frames alineados que produce la selección por ventana
(`equinos.secuencia` + `equinos.alineado`) y evalúa el clasificador de tres clases —`Disgustado`,
`Interesado`, `Sereno`— que sirve la interfaz.

Cubre el **objetivo específico 3** del plan de trabajo.

**Qué corre este notebook.** Reproduce la evaluación de `scripts/evaluar_v2.py` y la de
`scripts/evaluar_lineas_de_base.py`, y las muestra paso a paso. **No persiste pesos ni
resultados**: los únicos escritores de `objetivo3_evaluacion_v2.json`, de los pesos por fold y del
modelo de producción son esos dos scripts. El Paso 9 verifica que las predicciones de acá sean
idénticas a las persistidas.

## Qué hay que poder leer en este notebook

1. **La partición es por ejemplar**, nunca por clip, y la validación interna también. 29 de los
   133 caballos tienen clips en más de una clase; particionar por clip los pondría de los dos lados
   del corte. Eso no da error: da una métrica optimista y falsa (Paso 6).
2. **Las líneas de base van al lado del modelo**, sobre los mismos folds (Paso 5). Sin ellas un
   0,74 puede ser excelente o irrelevante.
3. **El resultado se informa aunque no convenga.**

## El resultado, por adelantado

**Modelo vigente: MIL multimodal, F1 macro 0,740 ± 0,052** sobre 359 clips de 133 caballos, con
4 folds por caballo, validación interna por caballo y *early stopping* sobre validación.

- **Supera con holgura a lo que no mira la cara**: clase mayoritaria 0,183, azar 0,325, y un
  predictor que sólo ve duración, fps y resolución del clip, 0,618. Ese último es el piso honesto:
  la forma en que se filmó cada clase ya separa bastante.
- **Empata con un promedio estático**: una regresión logística sobre la media de las
  características de los 16 frames da 0,741 ± 0,020. **No está demostrado que la atención, el
  orden de los frames ni el canal geométrico aporten** sobre un promedio. La hipótesis temporal no
  queda sostenida, igual que en la versión anterior de este trabajo (Bloque 4).
- **Sobreajusta el train** (F1 ≈ 0,98 contra ≈ 0,71 de validación), y el *early stopping* mantiene
  la estimación de test en línea con la validación.

## Paso 0 — Arranque: montar Drive y cargar el paquete `equinos`

In [1]:
# ============================================================
# ARRANQUE — persistencia en Google Drive
# ============================================================
# Autocontenida a propósito: corre ANTES de que el paquete `equinos` esté disponible.

import sys
from pathlib import Path

NOMBRE_PROYECTO = "tesina_equinos"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CODIGO = Path("/content/drive/MyDrive") / NOMBRE_PROYECTO / "codigo"
    if CODIGO.is_dir():
        sys.path.insert(0, str(CODIGO))
else:
    sys.path.insert(0, str(Path.cwd()))

from equinos import caracteristicas, clasificacion, entorno, lineas_de_base, particion, rutas

RAIZ = rutas.inicializar()

Entorno         : local
Raíz de proyecto: /media/sergio/Expansion/UGR/Licenciatura en Ciencias de datos/cuatrimestre3/materias/seminario trabajo final/Tesina/expresiones_faciales_equinos/datos_equinos
Área de trabajo : /media/sergio/Expansion/UGR/Licenciatura en Ciencias de datos/cuatrimestre3/materias/seminario trabajo final/Tesina/expresiones_faciales_equinos/datos_equinos/trabajo


## Paso 0a — Parámetros de la etapa

In [2]:
# Parámetros de la clasificación.
#
# Papermill inyecta una celda justo debajo de ésta cuando el notebook se ejecuta desde
# 00_pipeline.ipynb. Ejecutado a mano, valen estos valores por defecto.

# Folds de la validación cruzada. 4 y no 5: con 5 había folds con apenas 2 clips de `Disgustado` en
# test, y una métrica calculada sobre dos muestras no dice nada.
FOLDS = 4

# Los dos hiperparámetros del modelo vigente, elegidos por F1 de validación interna (nunca de test)
# entre dropout 0 / 0,2 / 0,3 y pesos de clase inversa / raíz. Son los de scripts/evaluar_v2.py.
DROPOUT = 0.3
PESOS = "raiz"

# True vuelve a extraer las características aunque la caché esté vigente. La caché se valida por
# huella contra el conjunto de clips, así que usarla es seguro.
REEXTRAER = False

# True cuando el notebook lo ejecuta 00_pipeline.ipynb: apaga las salidas largas.
EJECUCION_AUTOMATICA = False

## Paso 1 — Verificar el entorno

La etapa 03 **no** necesita DeepLabCut: sólo `torch`, `torchvision` y `scikit-learn`. Es la única
del pipeline que no arrastra la restricción de `numpy<2`.

In [3]:
entorno.verificar("modelado", gpu=True)

numpy 1.26.4 | pandas 2.3.3 | matplotlib 3.8.4 | scikit-learn 1.7.2 | torch 2.5.1+cu121 | torchvision 0.20.1+cu121
GPU: NVIDIA GeForce RTX 3070 Laptop GPU (8.4 GB) | torch 2.5.1+cu121


{'numpy': '1.26.4',
 'pandas': '2.3.3',
 'matplotlib': '3.8.4',
 'scikit-learn': '1.7.2',
 'torch': '2.5.1+cu121',
 'torchvision': '0.20.1+cu121'}

# Bloque 1: El conjunto de clips y la partición

## Paso 2 — Clips íntegros y partición por ejemplar

**La unidad es el clip íntegro**: sus 16 frames, elegidos dentro de la ventana temporal contigua
más larga (≤ 7 s) con medición del par ojo–hocico, cada uno anclado al frame mejor medido de su
entorno. De 393 clips candidatos entran 359 (91,3 %); los 34 restantes no tienen una ventana con
16 mediciones sin huecos de más de 0,5 s.

La partición se **persiste** y agrupa por `id_caballo`: todas las corridas —modelo, líneas de
base— leen los mismos cortes; comparar números medidos sobre cortes distintos no dice nada.

In [4]:
clips = particion.informe()

CONJUNTO DE CLIPS Y PARTICIÓN POR EJEMPLAR
Clips íntegros : 359
Ejemplares     : 133

Por clase:
  Disgustado    103 clips   57 ejemplares
  Interesado    118 clips   60 ejemplares
  Sereno        138 clips   52 ejemplares

Ejemplares con clips íntegros en MÁS DE UNA clase: 29
  022, 037, 044, 054, 055, 057, 154, 178, 201, 203, 204, 205, 208, 218, 220, 228, 229, 230, 231, 235, 237, 906, 907, 908, 909, 910, 912, 914, 916
  Por eso el grupo es el ejemplar y no el par ejemplar-clase: agrupar por el par los
  pondría de los dos lados del corte, y la métrica saldría optimista sin dar error.

Clips por detección propia (conjunto de la ablación): 359 de 359

Partición en 4 folds — sin fuga de ejemplar verificada
fold    clips  ejemplares   por clase
--------------------------------------------------------------
0          90          33   Disgustado 36  Interesado 29  Sereno 25
1          90          33   Disgustado 20  Interesado 28  Sereno 42
2          90          34   Disgustado 29  Inter

## Paso 3 — Verificar que el orden temporal es el que declara el CSV

Un desorden temporal no rompe nada: no lanza excepciones y todos los números siguen saliendo. Pero
**arruinaría la conclusión central de este notebook**: si los frames llegaran barajados de fábrica,
«permutar el orden no cambia las predicciones» sería trivialmente cierto y no diría nada.

La verificación es discriminativa y sin tolerancia absoluta: el vector de cada archivo tiene que
parecerse a su propia fila del tensor más que a cualquiera de las otras 15.

In [5]:
# Se comprueba sobre el dataset VIGENTE (`particion.ARCHIVO_FRAMES`, el de la selección por
# ventana), no sobre el de la corrida v1.
import csv
from collections import defaultdict

with (rutas.carpeta("resultados") / particion.ARCHIVO_FRAMES).open(encoding="utf-8") as f:
    filas = list(csv.DictReader(f))
por_clip = defaultdict(list)
for fila in filas:
    por_clip[fila["clip"]].append(fila)

integros = [c["clip"] for c in clips]
desordenados = [c for c in integros
                if sorted(int(f["orden"]) for f in por_clip[c]) != list(range(16))]
crecientes = [c for c in integros
              if [int(f["frame"]) for f in sorted(por_clip[c], key=lambda f: int(f["orden"]))]
              == sorted(int(f["frame"]) for f in por_clip[c])]
print(f"Clips con órdenes 0..15 completos     : {len(integros) - len(desordenados)} de {len(integros)}")
print(f"Clips cuyo orden sigue al del video   : {len(crecientes)} de {len(integros)}")
assert not desordenados, f"clips con orden incompleto: {desordenados[:5]}"
assert len(crecientes) == len(integros), "hay clips cuyo orden no sigue al tiempo del video"

Clips con órdenes 0..15 completos     : 359 de 359
Clips cuyo orden sigue al del video   : 359 de 359


# Bloque 2: Características del backbone congelado

## Paso 4 — Extraer y cachear

ResNet-18 preentrenada en ImageNet, **sin entrenar**, sin su capa clasificadora: 512 dimensiones
por frame, con la normalización fija de ImageNet (ninguna estadística se estima sobre los datos).

**Por qué congelado.** Descongelarlo agrega 11,2 millones de parámetros ajustables contra 359
clips. Sin él, lo que se entrena es la cabeza MIL: 132 355 parámetros, que ya alcanzan para
memorizar el train (Paso 12). Se probó además descongelar la última capa en la rama
`feat/finetuning-visual`: 0,747 ± 0,057, indistinguible (`docs/registro_de_experimentos.md`).

**Por qué se cachea.** Congelado, para un frame dado siempre devuelve el mismo vector. La caché se
valida por huella contra el conjunto de clips vigente.

In [6]:
if REEXTRAER:
    orden_clips, X = caracteristicas.extraer(clips)
else:
    try:
        orden_clips, X = caracteristicas.cargar()
        print(f"Características vigentes en caché: {len(X)} clips × {X[0].shape}")
    except (FileNotFoundError, ValueError) as motivo:
        print(f"{motivo}\nExtrayendo...")
        orden_clips, X = caracteristicas.extraer(clips)

indice = {c: i for i, c in enumerate(orden_clips)}

Características vigentes en caché: 359 clips × (16, 512)


# Bloque 3: Líneas de base

## Paso 5 — Todas, sobre los mismos folds

Sin ellas un F1 macro de 0,74 no significa nada. Se agrupan por lo que miran:

- **nada**: clase mayoritaria, y sólo `id_caballo`. Con partición por ejemplar el caballo de test
  nunca se vio, así que la de identidad cae al voto mayoritario y empata con la trivial — y **eso
  es la demostración de que la partición funciona**, no un defecto de la medición;
- **la filmación, no la cara**: duración sola, y metadatos (duración, fps, ancho, alto, ventana).
  Es el piso honesto para afirmar que el modelo lee la cara;
- **la cara, sin lo que el modelo agrega**: un frame central, la geometría sola, y el **promedio
  de los 16 frames**. Este último pone a prueba la atención, el orden y el canal geométrico.

Las lineales son regresiones logísticas con hiperparámetros fijados a priori y estandarización
ajustada sólo con el train de cada fold (`equinos/lineas_de_base.py`).

In [7]:
import numpy as np

from equinos import dataset as _ds
from equinos import pose

pose_filas = pose.leer_pose(rutas.carpeta("resultados") / _ds.ARCHIVO_POSE_SELECCIONADA)
folds = clasificacion.folds_por_ejemplar(clips)

mayoritaria = clasificacion.evaluar(clasificacion.predecir_mayoritaria, folds, "Clase mayoritaria")
identidad = clasificacion.evaluar(clasificacion.predecir_por_identidad, folds,
                                  "Sólo id_caballo, partición por ejemplar")

y = np.array([c["clase"] for c in clips])
fold = np.array([c["fold"] for c in clips])
E = lineas_de_base.entradas(clips, X, indice, pose_filas)
lineales = {
    "Solo duración del clip": E["duracion"],
    "Metadatos de filmación": E["metadatos"],
    "Geometría sola (media y desvío)": E["geometria"],
    "Visual, 1 frame central": E["visual_un_frame"],
    "Visual, media de 16 frames": E["visual_media"],
}
pred_base = {n: lineas_de_base.predecir_lineal(F, y, fold) for n, F in lineales.items()}
tabla = {n: lineas_de_base.resumen(y, p, fold) for n, p in pred_base.items()}
tabla["Clase mayoritaria"] = {"f1_macro": mayoritaria["f1_macro"][0], "f1_desvio": mayoritaria["f1_macro"][1]}
tabla["Sólo id_caballo"] = {"f1_macro": identidad["f1_macro"][0], "f1_desvio": identidad["f1_macro"][1]}
tabla["Azar estratificado (esperanza)"] = lineas_de_base.azar_estratificado(y, fold, simulaciones=500)

for nombre, r in sorted(tabla.items(), key=lambda kv: kv[1]["f1_macro"]):
    print(f"  {r['f1_macro']:.3f} ± {r['f1_desvio']:.3f}   {nombre}")

  0.183 ± 0.033   Clase mayoritaria
  0.183 ± 0.033   Sólo id_caballo
  0.324 ± 0.025   Azar estratificado (esperanza)
  0.544 ± 0.035   Solo duración del clip
  0.607 ± 0.033   Geometría sola (media y desvío)
  0.618 ± 0.034   Metadatos de filmación
  0.679 ± 0.024   Visual, 1 frame central
  0.741 ± 0.020   Visual, media de 16 frames


## Paso 6 — Cuánto vale el protocolo: la partición ingenua

La línea de base de identidad no puede contestar *¿la clase está determinada por el individuo?*
bajo un protocolo que impide que la identidad transfiera. Se mide con una partición **ingenua por
clip**, la que haría alguien que no agrupa.

Este número es un **diagnóstico del dataset**. No se reporta como resultado del trabajo.

In [8]:
ingenuo = clasificacion.imprimir(clasificacion.evaluar(
    clasificacion.predecir_por_identidad, clasificacion.folds_ingenuos(clips),
    "Sólo id_caballo, partición INGENUA por clip"))

fuga = ingenuo["f1_macro"][0] - identidad["f1_macro"][0]
clases_por_caballo = defaultdict(set)
for c in clips:
    clases_por_caballo[c["id_caballo"]].add(c["clase"])
una_sola = sum(1 for v in clases_por_caballo.values() if len(v) == 1)
print(f"\nFuga que la partición por ejemplar elimina: {fuga:+.3f} de F1 macro.")
print(f"La causa es una propiedad del dataset: {una_sola} de los {len(clases_por_caballo)} ejemplares")
print("aparecen en UNA sola clase, así que para ellos saber qué caballo es equivale a saber la clase.")


Sólo id_caballo, partición INGENUA por clip
  F1 macro           0.605 ± 0.044
  Balanced accuracy  0.605 ± 0.049
  Recall Disgustado  0.443 ± 0.101
  Cayó al voto mayoritario en 71 de 359 clips (20%): no había visto nunca a ese caballo
  por fold: 0.561  0.658  0.624  0.577

Fuga que la partición por ejemplar elimina: +0.421 de F1 macro.
La causa es una propiedad del dataset: 104 de los 133 ejemplares
aparecen en UNA sola clase, así que para ellos saber qué caballo es equivale a saber la clase.


## Paso 7 — El criterio de aceptación: el declarado y el re-declarado

*El modelo se considera útil si su F1 macro medio supera a la línea de base más alta en al menos un
desvío estándar entre folds.* Anclado en las líneas de base y no en un número redondo.

**El declarado antes de entrenar es 0,691** (`clasificacion.CRITERIO_ACEPTACION`): la línea de base
de un frame con voto, 0,626 ± 0,065, sobre el conjunto de entonces (141 clips de 73 caballos).

**Ese ancla quedó mal fundada** al cambiar el conjunto: sobre los 359 clips vigentes, la línea de
base más alta ya no es la de un frame sino el promedio de los 16 frames. Aplicar la misma regla al
conjunto vigente lo re-declara. Como exige la spec, se informa el resultado contra los dos, y se
deja escrito que la re-declaración se hizo **después** de ver los números (auditoría del
25/09/2026).

In [9]:
mas_alta = max((n for n in lineales), key=lambda n: tabla[n]["f1_macro"])
redeclarado = tabla[mas_alta]["f1_macro"] + tabla[mas_alta]["f1_desvio"]

print(f"DECLARADO antes de entrenar (conjunto de 141 clips) : {clasificacion.CRITERIO_ACEPTACION:.3f}")
print(f"Línea de base más alta del conjunto vigente         : {mas_alta}, "
      f"{tabla[mas_alta]['f1_macro']:.3f} ± {tabla[mas_alta]['f1_desvio']:.3f}")
print(f"RE-DECLARADO con la misma regla                     : {redeclarado:.3f}")

DECLARADO antes de entrenar (conjunto de 141 clips) : 0.691
Línea de base más alta del conjunto vigente         : Visual, media de 16 frames, 0.741 ± 0.020
RE-DECLARADO con la misma regla                     : 0.761


# Bloque 4: Cabezas temporales

## Paso 8 — Lo que se probó, y por qué no cambió nada

*Resultado histórico, sobre el conjunto de 141 clips de 73 caballos (28/08/2026). Se conserva
porque es la evidencia directa sobre el orden temporal; no se re-ejecuta acá.*

Se compararon **18 configuraciones**: tres cabezas —promedio, GRU, atención— en dos regímenes de
aumentación y dos de capacidad, con las muestras igualadas mediante ventanas de 8 frames.

| Cabeza | | Entrenables | F1 macro | Permutando el orden |
|---|---|---:|---:|---:|
| `promedio` | invariante | 33 027 | 0,574 ± 0,099 | 0,574 (+0,000) |
| `atencion` | sensible | 33 603 | **0,632 ± 0,155** | 0,632 (+0,000) |
| `gru` | sensible | 42 339 | 0,585 ± 0,114 | 0,619 (−0,034) |
| `promedio` chica | invariante | 4 131 | 0,587 ± 0,078 | 0,587 (+0,000) |
| `atencion` chica | sensible | 4 203 | 0,580 ± 0,098 | 0,580 (+0,000) |
| `gru` chica | sensible | 8 235 | 0,605 ± 0,072 | 0,613 (−0,008) |

**La atención da predicciones idénticas con los frames barajados**: el modelo *puede* usar el orden
y *no lo usa*. **La aumentación no movió nada** (−0,005 a +0,000): veinte vistas del mismo clip
siguen siendo el mismo caballo.

**El modelo vigente tampoco usa el orden, y por construcción**: la atención del MIL pondera los
frames y los suma, sin codificación de posición. Es un modelo de *conjunto* de frames, no de
secuencia. Por eso la prueba relevante hoy no es permutar, sino compararlo con un promedio
uniforme (Paso 5).

# Bloque 5: Modelo vigente y evaluación

## Paso 9 — Validación cruzada del MIL multimodal

`MILFusionModel`: por frame, las 512 características visuales se concatenan con un embedding de 128
de la geometría (ojo, hocico y oreja normalizados por el eje ojo–hocico, más máscaras de
validez); una atención pondera los 16 frames y un clasificador decide sobre la suma ponderada.

**Protocolo.** En cada fold externo, el resto se corta otra vez **por caballo** en train interno
(~93 caballos) y validación interna (7 caballos). La época se elige por F1 macro de validación
(paciencia 10, máximo 100) y el test se predice **una sola vez** con ese checkpoint.

No se persisten pesos (`pesos_en=None`): este paso reproduce, no reemplaza, a
`scripts/evaluar_v2.py`. Al final se verifica que las predicciones sean idénticas a las
persistidas, que son las que lee la interfaz.

In [10]:
import json

final = clasificacion.imprimir(clasificacion.evaluar(
    clasificacion.entrenar_mil_multimodal, folds, "MIL multimodal (vigente)",
    vectores=X, indice=indice, pose_filas=pose_filas, dropout=DROPOUT, estrategia_pesos=PESOS))

persistida = json.loads((rutas.carpeta("resultados") / clasificacion.ARCHIVO_EVALUACION)
                        .read_text(encoding="utf-8"))
esperadas = {p["clip"]: p["predicha"] for p in persistida["predicciones_fuera_de_muestra"]}
iguales = sum(1 for p in final["predicciones"] if esperadas.get(p["clip"]) == p["predicha"])
print(f"\nPredicciones idénticas a {clasificacion.ARCHIVO_EVALUACION}: {iguales} de {len(final['predicciones'])}")


MIL multimodal (vigente)
  F1 macro           0.740 ± 0.052
  Balanced accuracy  0.740 ± 0.051
  Recall Disgustado  0.690 ± 0.058
  por fold: 0.718  0.769  0.677  0.795

Predicciones idénticas a objetivo3_evaluacion_v2.json: 359 de 359


## Paso 10 — Contra el criterio: el declarado y el re-declarado

In [11]:
print("Contra el DECLARADO antes de entrenar:")
cumple_declarado = clasificacion.contra_el_criterio(final, criterio=clasificacion.CRITERIO_ACEPTACION)
print("\nContra el RE-DECLARADO sobre el conjunto vigente:")
cumple_redeclarado = clasificacion.contra_el_criterio(final, criterio=redeclarado)

Contra el DECLARADO antes de entrenar:

  criterio declarado (tarea 2.4) : 0.691
  F1 macro obtenido              : 0.740 ± 0.052
  CUMPLE  (diferencia +0.049)

Contra el RE-DECLARADO sobre el conjunto vigente:

  criterio declarado (tarea 2.4) : 0.761
  F1 macro obtenido              : 0.740 ± 0.052
  NO CUMPLE  (diferencia -0.021)


## Paso 11 — Matriz de confusión

Sobre las predicciones **fuera de muestra** agregadas de los 4 folds.

In [12]:
clasificacion.matriz_de_confusion(final["predicciones"])


MATRIZ DE CONFUSIÓN — fuera de muestra, agregada sobre los folds
real / predicha    Disgustado   Interesado       Sereno   total
Disgustado                 71           13           19     103
Interesado                  5           94           19     118
Sereno                     20           16          102     138
total                      96          123          140


## Paso 12 — Sobreajuste

En cada época se registra también el train interno, evaluado sin dropout. No hay un umbral
universal que declare sobreajuste: el gap se lee junto con las curvas y con que el test acompañe a
la validación. Las curvas completas están en la figura 5 (`04_figuras_tesis.ipynb`) y el análisis
en `docs/analisis_sobreajuste.md`.

*La ablación por `origen_alineacion` que ocupaba este paso dejó de tener objeto: con la selección
por ventana, los 359 clips se alinearon por detección propia (ninguno usa posiciones interpoladas
ni manuales), así que el subconjunto «sólo detectado» es el conjunto entero.*

In [13]:
print(f"Clips alineados sólo por detección propia: {sum(c['solo_detectado'] for c in clips)} de {len(clips)}\n")
print(f"{'fold':>4} {'época':>6} {'F1 train':>9} {'F1 val':>7} {'F1 test':>8} {'gap tr−val':>11} {'gap tr−test':>12}")
for f in final["por_fold"]:
    print(f"{f['fold']:>4} {f['mejor_epoca']:>6} {f['f1_train']:>9.3f} {f['f1_validacion']:>7.3f} "
          f"{f['f1_macro']:>8.3f} {f['f1_train'] - f['f1_validacion']:>11.3f} {f['f1_train'] - f['f1_macro']:>12.3f}")
media = lambda k: sum(f[k] for f in final["por_fold"]) / len(final["por_fold"])
print(f"\nmedia: F1 train {media('f1_train'):.3f}, validación {media('f1_validacion'):.3f}, test {media('f1_macro'):.3f}")

Clips alineados sólo por detección propia: 359 de 359

fold  época  F1 train  F1 val  F1 test  gap tr−val  gap tr−test
   0     28     0.973   0.700    0.718       0.273        0.254
   1     26     0.994   0.766    0.769       0.228        0.226
   2     20     0.969   0.641    0.677       0.329        0.292
   3     40     0.981   0.742    0.795       0.238        0.186

media: F1 train 0.979, validación 0.712, test 0.740


## Paso 13 — Intervalos por caballo y el confusor de filmación

Los 359 clips no son independientes: vienen de 244 videos y 133 caballos. El intervalo se calcula
remuestreando **caballos**, y las diferencias entre modelos se calculan pareadas.

Después, el control del confusor: se evalúan las mismas predicciones sólo sobre los clips
1080×1920 de al menos 5 s, que es donde la filmación deja de delatar la clase.

In [14]:
pred_mil = np.array([{p["clip"]: p["predicha"] for p in final["predicciones"]}[c["clip"]] for c in clips],
                    dtype=object)
caballos = np.array([c["id_caballo"] for c in clips])
b = lineas_de_base.bootstrap_por_caballo(
    y, {"MIL": pred_mil, "Visual, media de 16 frames": pred_base["Visual, media de 16 frames"],
        "Metadatos de filmación": pred_base["Metadatos de filmación"],
        "Visual, 1 frame central": pred_base["Visual, 1 frame central"]},
    caballos, referencia="MIL", replicas=1000)
for nombre, r in b["modelos"].items():
    linea = f"  IC95 F1 [{r['f1_ic95'][0]:.3f}; {r['f1_ic95'][1]:.3f}]  {nombre}"
    if "diferencia_referencia_menos_este" in r:
        d = r["diferencia_referencia_menos_este"]
        linea += f"   MIL − éste {d['media']:+.3f} [{d['ic95'][0]:+.3f}; {d['ic95'][1]:+.3f}]"
    print(linea)

mascara = lineas_de_base.sin_rasgos_delatores(clips, E["metadatos"])
print()
for nombre, p in (("MIL", pred_mil), ("Visual, media de 16 frames", pred_base["Visual, media de 16 frames"])):
    s = lineas_de_base.en_subconjunto(y, p, mascara)
    print(f"  sin rasgos delatores: {s['clips']} clips {s['clips_por_clase']} → "
          f"F1 {s['f1_macro']:.3f}, accuracy {s['accuracy']:.3f}  ({nombre})")

  IC95 F1 [0.678; 0.797]  MIL
  IC95 F1 [0.684; 0.796]  Visual, media de 16 frames   MIL − éste -0.004 [-0.066; +0.058]
  IC95 F1 [0.538; 0.694]  Metadatos de filmación   MIL − éste +0.121 [+0.031; +0.209]
  IC95 F1 [0.625; 0.735]  Visual, 1 frame central   MIL − éste +0.060 [+0.002; +0.118]

  sin rasgos delatores: 236 clips {'Disgustado': 28, 'Interesado': 80, 'Sereno': 128} → F1 0.705, accuracy 0.746  (MIL)
  sin rasgos delatores: 236 clips {'Disgustado': 28, 'Interesado': 80, 'Sereno': 128} → F1 0.671, accuracy 0.720  (Visual, media de 16 frames)


## Paso 14 — Errores por ejemplar

Si los fallos se concentran en pocos caballos, eso es información sobre el **dataset** —y sobre
las etiquetas—, no sobre el modelo.

In [15]:
clasificacion.errores_por_ejemplar(final["predicciones"])


ERRORES POR EJEMPLAR
  55 de 133 ejemplares fallan alguna vez (92 errores en total)
  los 10 peores concentran 37 (40%)
  906:6/13  013:5/5  037:5/14  053:4/4  044:3/3  251:3/3  904:3/4  910:3/9  916:3/3  026:2/4


# Bloque 6: Conclusión

## Paso 15 — Qué quedó demostrado y qué no

**Lo que sí quedó demostrado:**

- **Las imágenes tienen señal, y más de la que explica la filmación.** Con caballos que el modelo
  nunca vio, el F1 macro es 0,740 ± 0,052, contra 0,183 de la mayoritaria y 0,618 de un predictor
  que sólo ve duración, fps y resolución. La diferencia con este último excluye el cero en el
  bootstrap por caballo, y sobre los clips sin rasgos delatores el modelo conserva 0,705.
- **El confounder de individuo está cuantificado** (Paso 6), y la partición por ejemplar lo elimina.
- **La estimación es honesta por construcción**: partición y validación interna por caballo, época
  elegida sin mirar el test, predicciones idénticas en cada re-ejecución.

**Lo que no quedó demostrado:**

- **Que la arquitectura aporte.** Un promedio lineal de las características de los 16 frames rinde
  lo mismo (0,741 ± 0,020). Ni la atención, ni el orden temporal, ni el canal geométrico tienen un
  aporte demostrado sobre esa línea de base. La hipótesis temporal del trabajo no queda sostenida.
- **Que el modelo no memorice.** Sobreajusta el train (F1 ≈ 0,98); el *early stopping* contiene el
  efecto sobre el test, pero las probabilidades que muestra la interfaz están peor calibradas que
  sus decisiones.
- **Que generalice fuera de esta población**: no hay test externo, y la filmación correlaciona con
  la clase.
- **Que las etiquetas midan estados emocionales**: no hay protocolo de etiquetado documentado ni
  acuerdo entre anotadores (`docs/protocolo_de_etiquetado.md`).

## Qué haría falta para mejorarlo

1. **Más caballos, no más clips del mismo caballo.** La aumentación se probó y no movió nada.
2. **Filmar igual todas las clases**, o recortar todas las tomas a una duración común, para quitar
   el atajo de filmación.
3. **Un protocolo de etiquetado con doble anotación**, que diga cuál es el techo alcanzable.
4. Antes que más arquitectura: justificar la que hay contra el promedio estático, o adoptar el
   promedio por parsimonia.